In [ ]:
import warnings, requests, zipfile, io
warnings.simplefilter('ignore')
import pandas as pd
from scipy.io import arff
import boto3

In [ ]:
#set up data
bucket='c168894a4346736l11270733t1w825856716714-labbucket-pqll7a5zhgeg'

prefix='lab3'

train_file='vertebral_train.csv'
test_file='vertebral_test.csv'
validate_file='vertebral_validate.csv'

import os

s3_resource = boto3.Session().resource('s3')
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
#upload data
upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

# Train model

In [ ]:
import boto3
from sagemaker.image_uris import retrieve
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')


In [1]:
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

In [ ]:
import sagemaker
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)
xgb_model=sagemaker.estimator.Estimator(container,
                                       sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
xgb_model.fit(inputs=data_channels, logs=False)

# Hosting a model

In [ ]:
xgb_predictor = xgb_model.deploy(initial_instance_count=1,
                serializer = sagemaker.serializers.CSVSerializer(),
                instance_type='ml.m4.xlarge')

In [ ]:
row = test.iloc[0:1,1:]
row.head()

In [ ]:
batch_X_csv_buffer = io.StringIO()
row.to_csv(batch_X_csv_buffer, header=False, index=False)
test_row = batch_X_csv_buffer.getvalue()
print(test_row)

In [ ]:
xgb_predictor.predict(test_row)

# Batch transform for predictions

In [ ]:
batch_X = test.iloc[:,1:];
batch_X.head()

In [ ]:
batch_X_file='batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
#cut off class score
batch_X_file='batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = xgb_model.transformer(instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),sep=',',names=['class'])
target_predicted.head(5)

In [ ]:
def binary_convert(x):
    threshold = 0.65
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['binary'] = target_predicted['class'].apply(binary_convert)

print(target_predicted.head(10))
test.head(10)

# code implemented 

In [ ]:
def kaggle_feat_enginer(df):
    df.drop((df[df['person_age'] <= df["person_emp_length"]].index))
    df = df.drop(['id'], axis=1)
    #change {Y, N} values to {1, 0}
    df['cb_person_default_on_file'].replace({'Y': 1, 'N': 0}, inplace = True)
    df['loan_grade'].replace({'A': 6, 'B': 5, 'C': 4, 'D': 3,'E': 2, 'F': 1, 'G': 0}, inplace = True)
       
    #encode categorical variables
    df = df.join(pd.get_dummies(df['person_home_ownership']))
    df = df.join(pd.get_dummies(df['loan_intent']))
    df = df.drop(['loan_intent', 'person_home_ownership'], axis=1)
    return df

In [ ]:
train = pd.read_csv('train.csv')
validate = pd.read_csv('test.csv')

train = kaggle_feat_enginer(train)
validate = kaggle_feat_enginer(validate)

cols = ['loan_status'] + [col for col in test.columns if col != 'loan_status']

train, test = modsel.train_test_split(train, test_size = 0.2, random_state = 21, stratify = train['loan_status'])

train = train[cols]
test = test[cols]

In [ ]:
test = test.iloc[:,0:3]
train = train.iloc[:,0:3]
validate = validate.iloc[:,0:3]

In [ ]:
bucket='c168894a4346750l11270727t1w531891868-sandboxbucket-x1zf9xb6sb1p'

prefix='task4'

train_file= 'train.csv'
test_file = 'test.csv'
validate_file = 'validate.csv'

s3_resource = boto3.Session().resource('s3')
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
#upload data
upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

In [ ]:
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
#same model from task 3
hyperparams={"num_round":"100",
             'subsample':0.8,
             'colsample_bytree':0.7,
             "eval_metric": "auc",
             "objective": "binary:logistic"}

In [ ]:
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)
xgb_model=sagemaker.estimator.Estimator(container,
                                       sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix, validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
xgb_model.fit(inputs=data_channels, logs=False)